# SKlearnLM: train a small GPT from scratch

This notebook follows the **tokenization → next-token batches → causal transformer → loss → training → generation** path in the [reference notebook supplied by the user](https://drive.google.com/file/d/1qyydJT5PC0eK4B0pXlbJ3N1f3gPKA6i6/view). The 889-cell reference includes later pretrained-model and fine-tuning exercises; this adaptation implements the from-scratch path for the SKlearnLM documentation corpus.

The model starts with random weights. BPE learns merges from **training documents only**. Validation and test documents use the frozen tokenizer and remain separate throughout batching.

Run cells in order. Setup and tokenizer/model checks run first; set `RUN_TRAINING = True` in the configuration cell when ready to train. Set `RUN_TEST_EVALUATION = True` only for the final evaluation after choosing a checkpoint. No training results are pre-populated.

**Scope:** approximately two million documentation tokens is an educational corpus. Loss and generated examples must be measured; this notebook does not promise a reliable scikit-learn assistant.


## 1. Install missing packages and import libraries

Use Python 3.10+ in Jupyter or Google Colab. A CUDA GPU is optional. The cell installs missing libraries, without replacing an already installed PyTorch build. For CPU-only local installation, use PyTorch's [CPU wheel instructions](https://pytorch.org/get-started/locally/).

The notebook uses PyTorch's [causal scaled dot-product attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html) and Hugging Face [Tokenizers](https://huggingface.co/docs/tokenizers/).


In [ ]:
import importlib.util
import os
import subprocess
import sys

requirements = {
    "torch": "torch>=2.4,<3",
    "tokenizers": "tokenizers>=0.21,<1",
    "matplotlib": "matplotlib>=3.8,<4",
}
missing = [requirement for module, requirement in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    if os.environ.get("SKLEARNLM_NO_INSTALL") == "1":
        raise RuntimeError(f"Missing dependencies: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

from contextlib import nullcontext
import hashlib
import json
import math
from pathlib import Path
import random
import time
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Tokenizers:", __import__("tokenizers").__version__)


## 2. Configuration

The default model has four 256-dimensional transformer blocks, four attention heads, a context length of 256 and an 8,000-token vocabulary target. Adjust batch size first if GPU memory is limited.

`RUN_TRAINING` is off by default. The automated smoke mode uses a tiny CPU model and two optimizer updates to check execution; its losses are **not model-quality results**.


In [ ]:
SMOKE_TEST = os.environ.get("SKLEARNLM_SMOKE") == "1"
RUN_TRAINING = SMOKE_TEST  # Change to True to run the full configuration.
RUN_TEST_EVALUATION = SMOKE_TEST  # Enable once after checkpoint/model selection.
SEED = 42

CFG = {
    "vocab_size": 8000,
    "context_length": 256,
    "emb_dim": 256,
    "n_heads": 4,
    "n_layers": 4,
    "dropout": 0.1,
    "batch_size": 8,
    "grad_accum": 4,
    "epochs": 5,
    "learning_rate": 3e-4,
    "minimum_lr": 3e-5,
    "weight_decay": 0.1,
    "warmup_fraction": 0.05,
    "eval_every": 50,
    "eval_batches": 20,  # A validation probe during training; final validation is full.
    "patience": 5,  # Evaluation checks without improvement, not epochs.
    "max_steps": None,
}
if SMOKE_TEST:
    CFG.update(vocab_size=512, context_length=32, emb_dim=64, n_heads=4,
               n_layers=2, dropout=0.0, batch_size=2, grad_accum=1,
               epochs=1, eval_every=1, eval_batches=2, max_steps=2, patience=10)

BPE_VOCAB_TARGET = CFG["vocab_size"]

DEVICE = "cpu" if SMOKE_TEST else ("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if SMOKE_TEST:
    torch.set_num_threads(2)

DATA_REVISION = "bb503b2b994518c4640e4d966392b03cb35b6cbe"
DATA_BASE_URL = f"https://raw.githubusercontent.com/vishal88736/SKlearnLM/{DATA_REVISION}/data/"
EOS = "<|endoftext|>"
print("Device:", DEVICE)
print(json.dumps(CFG, indent=2))


## 3. Load the verified documentation splits

An existing local `data/` folder is used when available. When the notebook runs alone in Colab, it downloads the manifest and three text files from a pinned SKlearnLM commit. Each text file must match the manifest's byte length and SHA-256 before use.

Training has 936 pages, validation 90, and test 49. Topic families are disjoint; generic API prose can still overlap, as documented in the dataset audit. No other text dataset is loaded.


In [ ]:
def sha256(value):
    return hashlib.sha256(value).hexdigest()

override = os.environ.get("SKLEARNLM_DATA_DIR")
candidates = ([Path(override)] if override else
              [Path.cwd() / "data", Path.cwd().parent / "data"])
DATA_DIR = (Path(override) if override else
            next((path for path in candidates if (path / "manifest.json").is_file()),
                 Path.cwd() / "sklearnlm_data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_DIR = DATA_DIR.parent
RUN_DIR = Path(os.environ.get("SKLEARNLM_RUN_DIR", str(PROJECT_DIR / "runs" / "sklearnlm")))
RUN_DIR.mkdir(parents=True, exist_ok=True)

def fetch_missing(name):
    target = DATA_DIR / name
    if not target.exists():
        request = Request(DATA_BASE_URL + name, headers={"User-Agent": "SKlearnLM-notebook"})
        with urlopen(request, timeout=60) as response:
            payload = response.read()
        temporary = target.with_suffix(target.suffix + ".download")
        temporary.write_bytes(payload)
        temporary.replace(target)
    return target

manifest_bytes = fetch_missing("manifest.json").read_bytes()
manifest = json.loads(manifest_bytes)
assert manifest["documentation_version"] == "1.10.dev0"
assert manifest["document_boundary_token"] == EOS
DATA_MANIFEST_SHA = sha256(manifest_bytes)

texts = {}
for split in ("train", "validation", "test"):
    name = split + ".txt"
    payload = fetch_missing(name).read_bytes()
    expected = manifest["files"][name]
    if len(payload) != expected["bytes"] or sha256(payload) != expected["sha256"]:
        raise ValueError(f"Integrity check failed for {name}; restore the matching dataset snapshot.")
    texts[split] = payload.decode("utf-8")
    expected_docs = manifest["statistics"]["splits"][split]["documents"]
    assert texts[split].count(EOS) == expected_docs
    print(f"{split:10s}: {len(payload):,} bytes, {expected_docs} documents")

(RUN_DIR / "data_manifest.json").write_bytes(manifest_bytes)
print("Artifacts:", RUN_DIR.resolve())


## 4. Train a byte-level BPE tokenizer on training documents

All 256 byte values are included, so unseen words and Unicode text can be encoded without an unknown-word replacement. Case, Python indentation and original whitespace are retained. `<|endoftext|>` is a registered special token.

BPE learns from `texts["train"]` only. Counts below are measured with this tokenizer and can differ from earlier corpus estimates.


In [ ]:
tokenizer_path = RUN_DIR / "tokenizer.json"
tokenizer_metadata_path = RUN_DIR / "tokenizer_metadata.json"
tokenizer_settings = {
    "trained_on": "train.txt",
    "training_file_sha256": manifest["files"]["train.txt"]["sha256"],
    "requested_vocabulary": BPE_VOCAB_TARGET,
    "minimum_frequency": 2,
    "special_token": EOS,
    "pretokenizer": "ByteLevel(add_prefix_space=False)",
}
if tokenizer_path.exists() and tokenizer_metadata_path.exists():
    saved_settings = json.loads(tokenizer_metadata_path.read_text())
    if saved_settings != tokenizer_settings:
        raise ValueError("Saved tokenizer settings differ. Choose a new RUN_DIR for this experiment.")
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    print("Loaded the matching frozen training tokenizer.")
else:
    if any((RUN_DIR / name).exists() for name in ("best.pt", "last.pt")):
        raise ValueError("Checkpoints exist without matching tokenizer metadata. Choose a new RUN_DIR.")
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=BPE_VOCAB_TARGET,
        min_frequency=2,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        special_tokens=[EOS],
        show_progress=not SMOKE_TEST,
    )
    train_documents = [document for document in texts["train"].split(EOS) if document.strip()]
    tokenizer.train_from_iterator(train_documents, trainer=trainer, length=len(train_documents))
    tokenizer.save(str(tokenizer_path))
    tokenizer_metadata_path.write_text(json.dumps(tokenizer_settings, indent=2) + "\n")
EOS_ID = tokenizer.token_to_id(EOS)
assert EOS_ID is not None
CFG["vocab_size"] = tokenizer.get_vocab_size()
TOKENIZER_SHA = sha256(tokenizer_path.read_bytes())

roundtrip_sample = "def square(x):\n    return x ** 2\n\n# Unicode: Δ, café\n" + EOS
assert tokenizer.decode(tokenizer.encode(roundtrip_sample).ids, skip_special_tokens=False) == roundtrip_sample
serialized_tokenizer = Tokenizer.from_file(str(tokenizer_path))
assert serialized_tokenizer.encode(roundtrip_sample).ids == tokenizer.encode(roundtrip_sample).ids

token_ids = {}
for split, text in texts.items():
    ids = tokenizer.encode(text).ids
    assert ids.count(EOS_ID) == manifest["statistics"]["splits"][split]["documents"]
    token_ids[split] = torch.tensor(ids, dtype=torch.long)
    print(f"{split:10s}: {len(ids):,} tokens")
print("Vocabulary:", CFG["vocab_size"], "EOS ID:", EOS_ID)
print("Whitespace/Unicode and tokenizer serialization round trips: passed")


## 5. Form next-token input/target pairs

For input tokens `[t0, t1, ..., t255]`, the target is `[t1, t2, ..., t256]`. Windows do not overlap within a split; at most one context-sized tail is omitted. Windows may contain adjacent documents from the same split, separated by EOS.

Only training windows are shuffled. Separate DataLoaders prevent validation/test tokens from entering the optimization loop.


In [ ]:
class TokenWindows(Dataset):
    def __init__(self, ids, context_length):
        self.ids = ids
        self.context_length = context_length
        self.count = max(0, (len(ids) - 1) // context_length)
        if not self.count:
            raise ValueError("Split is shorter than one context window.")

    def __len__(self):
        return self.count

    def __getitem__(self, index):
        start = index * self.context_length
        stop = start + self.context_length
        return self.ids[start:stop], self.ids[start + 1:stop + 1]

datasets = {split: TokenWindows(ids, CFG["context_length"])
            for split, ids in token_ids.items()}
generator = torch.Generator().manual_seed(SEED)
common_loader_args = dict(batch_size=CFG["batch_size"], num_workers=0,
                          pin_memory=(DEVICE == "cuda"))
train_loader = DataLoader(datasets["train"], shuffle=True, drop_last=True,
                          generator=generator, **common_loader_args)
train_eval_loader = DataLoader(datasets["train"], shuffle=False, **common_loader_args)
validation_loader = DataLoader(datasets["validation"], shuffle=False, **common_loader_args)
test_loader = DataLoader(datasets["test"], shuffle=False, **common_loader_args)
assert len(train_loader) > 0
x_example, y_example = datasets["train"][0]
assert torch.equal(x_example[1:], y_example[:-1])
print("Input shape:", tuple(x_example.shape), "target shape:", tuple(y_example.shape))
print("Training batches per epoch:", len(train_loader))


## 6. Build the causal transformer

The reference's building blocks are retained: token and learned positional embeddings, causal multi-head self-attention, two pre-normalized residual paths, a four-times-expanded GELU feed-forward network, and a final vocabulary projection.

This implementation uses fused Q/K/V projection and PyTorch's causal attention primitive. Input embeddings and the output vocabulary projection share weights. No pretrained weights are loaded.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        width = cfg["emb_dim"]
        assert width % cfg["n_heads"] == 0
        self.n_heads = cfg["n_heads"]
        self.head_dim = width // self.n_heads
        self.dropout = cfg["dropout"]
        self.qkv = nn.Linear(width, 3 * width, bias=False)
        self.out_proj = nn.Linear(width, width)

    def forward(self, x):
        batch, length, width = x.shape
        qkv = self.qkv(x).view(batch, length, 3, self.n_heads, self.head_dim)
        queries, keys, values = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        attended = F.scaled_dot_product_attention(
            queries, keys, values,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        attended = attended.transpose(1, 2).contiguous().view(batch, length, width)
        return self.out_proj(attended)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        width = cfg["emb_dim"]
        self.norm1 = nn.LayerNorm(width)
        self.norm2 = nn.LayerNorm(width)
        self.attention = CausalSelfAttention(cfg)
        self.feed_forward = nn.Sequential(
            nn.Linear(width, 4 * width),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * width, width),
        )
        self.residual_dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        x = x + self.residual_dropout(self.attention(self.norm1(x)))
        return x + self.residual_dropout(self.feed_forward(self.norm2(x)))


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = dict(cfg)
        width = cfg["emb_dim"]
        self.token_embedding = nn.Embedding(cfg["vocab_size"], width)
        self.position_embedding = nn.Embedding(cfg["context_length"], width)
        self.embedding_dropout = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = nn.LayerNorm(width)
        self.lm_head = nn.Linear(width, cfg["vocab_size"], bias=False)
        self.apply(self.initialize)
        self.lm_head.weight = self.token_embedding.weight

    @staticmethod
    def initialize(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if getattr(module, "bias", None) is not None:
                nn.init.zeros_(module.bias)

    def forward(self, input_ids, targets=None):
        _, length = input_ids.shape
        if length > self.cfg["context_length"]:
            raise ValueError("Input exceeds configured context length.")
        positions = torch.arange(length, device=input_ids.device)
        x = self.embedding_dropout(
            self.token_embedding(input_ids) + self.position_embedding(positions)
        )
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.final_norm(x))
        loss = None if targets is None else F.cross_entropy(
            logits.reshape(-1, logits.size(-1)), targets.reshape(-1)
        )
        return logits, loss


model = GPTModel(CFG).to(DEVICE)
PARAMETERS = sum(parameter.numel() for parameter in model.parameters())
assert model.lm_head.weight is model.token_embedding.weight
print(f"Trainable parameters: {PARAMETERS:,}")
print("Weight initialization: random; output/input embeddings tied")


## 7. Check target alignment, causal masking and gradients

Changing later tokens must not change earlier logits. A forward/backward pass must produce finite loss and gradients. These checks validate model wiring; they do not measure learned language quality.


In [ ]:
check_length = min(32, CFG["context_length"])
check_x = token_ids["train"][:check_length].unsqueeze(0).to(DEVICE)
check_y = token_ids["train"][1:check_length + 1].unsqueeze(0).to(DEVICE)
model.eval()
with torch.no_grad():
    original_logits, initial_loss = model(check_x, check_y)
    changed = check_x.clone()
    cut = check_length // 2
    changed[:, cut:] = (changed[:, cut:] + 1) % CFG["vocab_size"]
    changed_logits, _ = model(changed)
    assert torch.allclose(original_logits[:, :cut], changed_logits[:, :cut],
                          rtol=1e-5, atol=1e-5), "Future-token leakage"
assert tuple(original_logits.shape) == (1, check_length, CFG["vocab_size"])
assert torch.isfinite(initial_loss)

model.train()
_, check_loss = model(check_x, check_y)
check_loss.backward()
assert model.token_embedding.weight.grad is not None
assert all(torch.isfinite(parameter.grad).all()
           for parameter in model.parameters() if parameter.grad is not None)
model.zero_grad(set_to_none=True)
print("Shifted targets, causal mask, logits shape and finite gradients: passed")
print(f"Random-initialization loss on one training window: {initial_loss.item():.4f}")


## 8. Evaluation, optimizer settings and checkpoint helpers

Cross-entropy is averaged by target-token count. Training logs use a fixed validation probe; the selected checkpoint receives a full validation pass later. Test loss is gated separately.

CUDA uses bfloat16 when supported. Other devices use float32. Checkpoints include the model, optimizer, configuration, tokenizer fingerprint and dataset fingerprint.


In [ ]:
USE_AMP = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16
USE_SCALER = False  # bfloat16 and float32 do not need gradient scaling.

def autocast_context():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE) if USE_AMP else nullcontext()


@torch.inference_mode()
def evaluate(loader, max_batches=None):
    was_training = model.training
    model.eval()
    total_loss, target_tokens = 0.0, 0
    try:
        for batch_index, (x, y) in enumerate(loader):
            if max_batches is not None and batch_index >= max_batches:
                break
            x, y = x.to(DEVICE), y.to(DEVICE)
            with autocast_context():
                _, loss = model(x, y)
            total_loss += float(loss) * y.numel()
            target_tokens += y.numel()
    finally:
        model.train(was_training)
    if not target_tokens:
        raise ValueError("Evaluation received no target tokens.")
    return total_loss / target_tokens


def learning_rate_at(step, total_steps):
    warmup = max(1, int(total_steps * CFG["warmup_fraction"]))
    if step < warmup:
        return CFG["learning_rate"] * (step + 1) / warmup
    progress = min(1.0, (step - warmup) / max(1, total_steps - warmup - 1))
    return CFG["minimum_lr"] + 0.5 * (CFG["learning_rate"] - CFG["minimum_lr"]) * (
        1 + math.cos(math.pi * progress)
    )


def save_checkpoint(name, optimizer, scaler, step, history, tokens_seen):
    payload = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "amp_scaler": scaler.state_dict(),
        "config": dict(CFG),
        "step": step,
        "history": history,
        "tokens_seen": tokens_seen,
        "tokenizer_sha256": TOKENIZER_SHA,
        "data_manifest_sha256": DATA_MANIFEST_SHA,
        "seed": SEED,
        "smoke_test": SMOKE_TEST,
        "torch_version": str(torch.__version__),
    }
    target = RUN_DIR / name
    temporary = target.with_suffix(".tmp")
    torch.save(payload, temporary)
    temporary.replace(target)
    return target


def load_model_checkpoint(path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if checkpoint["tokenizer_sha256"] != TOKENIZER_SHA:
        raise ValueError("Checkpoint was created with a different tokenizer.")
    if checkpoint["data_manifest_sha256"] != DATA_MANIFEST_SHA:
        raise ValueError("Checkpoint was created with different dataset files.")
    for key in ("vocab_size", "context_length", "emb_dim", "n_heads", "n_layers"):
        if checkpoint["config"][key] != CFG[key]:
            raise ValueError(f"Model configuration differs at {key}.")
    model.load_state_dict(checkpoint["model"])
    return checkpoint


## 9. Train with validation checkpoints

Set `RUN_TRAINING = True` in the configuration cell and run the notebook from there. The loop uses AdamW, gradient accumulation, warmup/cosine learning-rate scheduling, gradient clipping and validation-based early stopping.

A final partial accumulation window is scaled by its actual batch count. Training uses only `train_loader`. The best validation checkpoint and latest optimizer state are saved separately.

The history records `validation_loss_probe` because only a subset of validation batches is used during training. Test data is not used to select a checkpoint.


In [ ]:
def train_model():
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["learning_rate"],
                                 betas=(0.9, 0.95), weight_decay=CFG["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    updates_per_epoch = math.ceil(len(train_loader) / CFG["grad_accum"])
    total_steps = CFG["epochs"] * updates_per_epoch
    if CFG["max_steps"] is not None:
        total_steps = min(total_steps, CFG["max_steps"])
    if total_steps < 1:
        raise ValueError("No optimizer updates configured.")

    history, step, tokens_seen = [], 0, 0
    best_validation, stale_checks = float("inf"), 0
    started = time.perf_counter()
    stop = False
    model.train()
    optimizer.zero_grad(set_to_none=True)
    accumulated_loss = accumulated_tokens = 0

    for epoch in range(CFG["epochs"]):
        for batch_index, (x, y) in enumerate(train_loader):
            x, y = x.to(DEVICE), y.to(DEVICE)
            window_start = (batch_index // CFG["grad_accum"]) * CFG["grad_accum"]
            window_batches = min(CFG["grad_accum"], len(train_loader) - window_start)
            with autocast_context():
                _, loss = model(x, y)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite training loss")
            scaler.scale(loss / window_batches).backward()
            accumulated_loss += float(loss.detach()) * y.numel()
            accumulated_tokens += y.numel()
            tokens_seen += y.numel()

            update = ((batch_index + 1) % CFG["grad_accum"] == 0
                      or batch_index + 1 == len(train_loader))
            if not update:
                continue
            lr = learning_rate_at(step, total_steps)
            for group in optimizer.param_groups:
                group["lr"] = lr
            scaler.unscale_(optimizer)
            grad_norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0,
                                                error_if_nonfinite=True)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            step += 1

            check = (step == 1 or step % CFG["eval_every"] == 0
                     or step == total_steps or batch_index + 1 == len(train_loader))
            if check:
                validation_loss = evaluate(validation_loader, CFG["eval_batches"])
                if not math.isfinite(validation_loss):
                    raise FloatingPointError("Non-finite validation loss")
                row = {
                    "step": step, "epoch": epoch + 1, "tokens_seen": tokens_seen,
                    "train_loss": accumulated_loss / accumulated_tokens,
                    "validation_loss_probe": validation_loss, "learning_rate": lr,
                    "gradient_norm": float(grad_norm),
                    "elapsed_seconds": time.perf_counter() - started,
                }
                history.append(row)
                print(f"step {step:4d}/{total_steps} | train {row['train_loss']:.4f} | "
                      f"validation probe {validation_loss:.4f} | lr {lr:.2e}")
                accumulated_loss = accumulated_tokens = 0
                if validation_loss < best_validation - 1e-4:
                    best_validation, stale_checks = validation_loss, 0
                    save_checkpoint("best.pt", optimizer, scaler, step, history, tokens_seen)
                else:
                    stale_checks += 1
                save_checkpoint("last.pt", optimizer, scaler, step, history, tokens_seen)
                (RUN_DIR / "history.json").write_text(json.dumps(history, indent=2) + "\n")
                if stale_checks >= CFG["patience"]:
                    print("Early stopping after validation checks without improvement.")
                    stop = True
            if step >= total_steps:
                stop = True
            if stop:
                break
        if stop:
            break

    save_checkpoint("last.pt", optimizer, scaler, step, history, tokens_seen)
    return history


history = []
TRAINED_THIS_SESSION = False
if RUN_TRAINING:
    history = train_model()
    TRAINED_THIS_SESSION = True
    print("Completed optimizer updates:", history[-1]["step"])
else:
    print("Training is disabled. Set RUN_TRAINING = True in the configuration cell.")


## 10. Inspect the learning curves

Compare training loss with the validation probe. Rising validation loss while training loss falls is a sign of memorization. Smoke-mode plots are execution checks only.


In [ ]:
if history:
    steps = [row["step"] for row in history]
    plt.figure(figsize=(8, 4))
    plt.plot(steps, [row["train_loss"] for row in history], label="training")
    plt.plot(steps, [row["validation_loss_probe"] for row in history],
             label="validation probe")
    plt.xlabel("Optimizer updates")
    plt.ylabel("Cross-entropy loss (nats/token)")
    plt.title("CPU smoke check" if SMOKE_TEST else "SKlearnLM training")
    plt.legend()
    plt.tight_layout()
    plt.savefig(RUN_DIR / "loss_curve.png", dpi=150)
    plt.show()
else:
    print("No training history yet.")


## 11. Restore the best checkpoint and evaluate validation

The model is restored using its tokenizer and dataset fingerprints. A second model instance checks that the saved weights reproduce the restored model's logits.

A normal training run evaluates the entire validation split here. Smoke mode evaluates only two batches and labels that limited scope explicitly.


In [ ]:
HAS_TRAINED_CHECKPOINT = False
metrics = {}
best_path = RUN_DIR / "best.pt"
if best_path.exists():
    checkpoint = load_model_checkpoint(best_path)
    assert checkpoint["step"] > 0
    if bool(checkpoint["smoke_test"]) != SMOKE_TEST:
        raise ValueError("Smoke/full-run checkpoint mismatch; choose a separate RUN_DIR.")
    HAS_TRAINED_CHECKPOINT = True
    model.eval()
    restored_copy = GPTModel(CFG).to(DEVICE)
    restored_copy.load_state_dict(checkpoint["model"])
    restored_copy.eval()
    with torch.no_grad():
        expected_logits, _ = model(check_x)
        restored_logits, _ = restored_copy(check_x)
        assert torch.allclose(expected_logits, restored_logits, rtol=1e-5, atol=1e-5)
    del restored_copy
    evaluation_limit = 2 if SMOKE_TEST else None
    validation_loss = evaluate(validation_loader, evaluation_limit)
    metrics = {
        "smoke_test": SMOKE_TEST,
        "checkpoint_step": checkpoint["step"],
        "parameters": PARAMETERS,
        "data_manifest_sha256": DATA_MANIFEST_SHA,
        "tokenizer_sha256": TOKENIZER_SHA,
        "token_counts": {split: len(ids) for split, ids in token_ids.items()},
        "validation": {
            "cross_entropy": validation_loss,
            "perplexity": math.exp(validation_loss),
            "scope": "two_batches_only" if SMOKE_TEST else "full_split_windows",
        },
    }
    print("Checkpoint round trip: passed")
    print(json.dumps(metrics["validation"], indent=2))
else:
    print("No trained checkpoint exists yet. Run training first.")


## 12. Final test evaluation

Enable `RUN_TEST_EVALUATION` after choosing the model with validation data. The test split must not be used to choose hyperparameters or an early-stopping point. Perplexity describes next-token prediction under this tokenizer; it is not factual accuracy or a chatbot score.


In [ ]:
if HAS_TRAINED_CHECKPOINT and RUN_TEST_EVALUATION:
    test_loss = evaluate(test_loader, 2 if SMOKE_TEST else None)
    metrics["test"] = {
        "cross_entropy": test_loss,
        "perplexity": math.exp(test_loss),
        "scope": "two_batches_only" if SMOKE_TEST else "full_split_windows",
    }
    print(json.dumps(metrics["test"], indent=2))
else:
    print("Test evaluation is disabled or no trained checkpoint is available.")
if metrics:
    (RUN_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")


## 13. Generate a documentation continuation

The prompt below starts a documentation page. The model predicts a continuation; it has not been instruction-tuned. Treat generated API details as unverified text and compare them with the official documentation.


In [ ]:
@torch.inference_mode()
def generate_text(prompt, max_new_tokens=160, temperature=0.8, top_k=40):
    if not prompt:
        raise ValueError("Prompt must contain text.")
    if temperature < 0 or (top_k is not None and top_k < 1):
        raise ValueError("Invalid sampling settings.")
    ids = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long, device=DEVICE)
    was_training = model.training
    model.eval()
    try:
        for _ in range(max_new_tokens):
            context = ids[:, -CFG["context_length"]:]
            logits, _ = model(context)
            logits = logits[:, -1, :]
            if top_k is not None:
                cutoff = torch.topk(logits, min(top_k, logits.shape[-1])).values[:, -1:]
                logits = logits.masked_fill(logits < cutoff, float("-inf"))
            if temperature == 0:
                next_id = logits.argmax(dim=-1, keepdim=True)
            else:
                next_id = torch.multinomial(F.softmax(logits / temperature, dim=-1), 1)
            if next_id.item() == EOS_ID:
                break
            ids = torch.cat([ids, next_id], dim=1)
    finally:
        model.train(was_training)
    return tokenizer.decode(ids[0].tolist(), skip_special_tokens=False)


if HAS_TRAINED_CHECKPOINT:
    torch.manual_seed(SEED)
    prompt = "# LinearRegression\n\n"
    sample_text = generate_text(prompt, max_new_tokens=12 if SMOKE_TEST else 160)
    assert isinstance(sample_text, str) and sample_text.startswith(prompt)
    print("SMOKE OUTPUT: not a trained-quality example" if SMOKE_TEST else "Generated continuation:")
    print(sample_text)
    (RUN_DIR / "sample.txt").write_text(sample_text, encoding="utf-8")
else:
    print("Generation waits for a trained checkpoint.")


## 14. Save a portable inference bundle

The ZIP includes model weights/configuration, the matching tokenizer, dataset metadata and measured metrics. Optimizer checkpoints (`best.pt` and `last.pt`) stay in `RUN_DIR`; they are not placed in the smaller inference bundle.

Keep the model architecture cells or this notebook alongside the ZIP when reloading it. On Colab, download the ZIP from the Files panel after training.


In [ ]:
import zipfile

if HAS_TRAINED_CHECKPOINT:
    inference_payload = {
        key: checkpoint[key] for key in (
            "model", "config", "step", "tokenizer_sha256", "data_manifest_sha256",
            "seed", "smoke_test", "torch_version",
        )
    }
    inference_path = RUN_DIR / "inference_model.pt"
    torch.save(inference_payload, inference_path)
    (RUN_DIR / "config.json").write_text(json.dumps(CFG, indent=2) + "\n")
    archive_path = RUN_DIR / "sklearnlm_inference.zip"
    names = ["inference_model.pt", "tokenizer.json", "tokenizer_metadata.json",
             "config.json", "data_manifest.json", "metrics.json"]
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in names:
            path = RUN_DIR / name
            assert path.is_file(), f"Missing export artifact: {name}"
            archive.write(path, arcname=name)
    print("Inference bundle:", archive_path.resolve())
    print("Optimizer checkpoints:", (RUN_DIR / "best.pt").resolve(),
          (RUN_DIR / "last.pt").resolve())
else:
    print("Export waits for a trained checkpoint.")


## Interpretation and reproducibility

- The dataset, tokenizer and model configuration are fingerprinted in the saved artifacts.
- BPE and neural weights are learned only from SKlearnLM training text.
- Validation selects the best checkpoint. The optional final test evaluates a separate set of topic families.
- Shared API boilerplate remains in the source corpus; disjoint group IDs do not prove semantic independence.
- Smoke mode checks a two-update CPU execution path, not convergence, GPU throughput or model quality.
- Full training is controlled by the configuration cell. Runtime and quality depend on the device, tokenizer, optimization settings and corpus.

**Reference:** user-supplied Google Drive notebook, SHA-256 `e8e1fdd3538abb436727a1d2cb3d186f44a0badafc731693bcd379af634c5768` (889 cells). Its from-scratch attention/model/loss/training/generation sections informed the organization. This notebook uses newly written code and explanations; the full reference is not redistributed. The reference points to Sebastian Raschka's [LLMs from Scratch](https://github.com/rasbt/LLMs-from-scratch).
